# Modelos Jerárquicos Bayesianos
### Análisis en Python — Capítulos 15, 16 y 17 de *Bayes Rules!*

**Librerías necesarias:**
```
pip install pymc arviz pandas numpy matplotlib scipy
```

Este cuaderno replica la estructura de los ejemplos del libro usando datos sintéticos que reproducen las propiedades estadísticas de los datasets originales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as stats
import pymc as pm
import arviz as az

# Reproducibilidad
SEED = 42
rng  = np.random.default_rng(SEED)

# Estilo visual oscuro consistente con la presentación
plt.style.use('dark_background')
PURPLE = '#8b5cf6'
BLUE   = '#38bdf8'
GREEN  = '#34d399'
ORANGE = '#fb923c'
WHITE  = '#f0f0f8'

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

---
## PARTE 0: Ejemplo principal — Turismo en Colombia
### El hilo conductor de toda la presentación

**Contexto:**
- 3 ciudades: Cartagena, Medellín, Bogotá — **5 turistas por ciudad**
- 1 ciudad nueva sin datos: **Cali**
- Variable respuesta: gasto turístico (miles de COP)
- Predictor: días de estadía
- **Pregunta:** ¿Cómo estimo el gasto esperado de un turista en cada ciudad?


In [ ]:
# ── Datos sintéticos: turismo Colombia ───────────────────────────────────────
np.random.seed(SEED)

ciudades = ['Cartagena', 'Medellín', 'Bogotá']
CIUDAD_COLORS = {'Cartagena': BLUE, 'Medellín': PURPLE, 'Bogotá': GREEN}
N_POR_CIUDAD = 5

# Parámetros verdaderos por ciudad (desconocidos en la práctica)
params_verdaderos = {
    'Cartagena': {'mu': 950,  'beta': 160, 'sigma': 120},  # playa, cara
    'Medellín':  {'mu': 600,  'beta': 110, 'sigma': 90},   # moda / city trip
    'Bogotá':    {'mu': 420,  'beta': 70,  'sigma': 100},  # negocios / tránsito
}

dias_data, gasto_data, ciudad_data = [], [], []
for ciudad in ciudades:
    p = params_verdaderos[ciudad]
    dias  = np.random.randint(2, 8, N_POR_CIUDAD).astype(float)
    gasto = p['mu'] + p['beta'] * dias + np.random.normal(0, p['sigma'], N_POR_CIUDAD)
    gasto = np.clip(gasto, 100, None)
    dias_data.extend(dias)
    gasto_data.extend(gasto)
    ciudad_data.extend([ciudad] * N_POR_CIUDAD)

turismo = pd.DataFrame({'ciudad': ciudad_data, 'dias': dias_data, 'gasto': gasto_data})
turismo['ciudad_id'] = pd.Categorical(turismo['ciudad'], categories=ciudades).codes
N_CIUDADES = len(ciudades)
ciudad_idx = turismo['ciudad_id'].values.astype(int)

print(turismo.to_string(index=False))
print(f'\nMedia por ciudad:')
print(turismo.groupby('ciudad')[['dias','gasto']].mean().round(1))


In [ ]:
# ── Exploración visual ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for ciudad in ciudades:
    subset = turismo[turismo['ciudad'] == ciudad]
    ax.scatter(subset['dias'], subset['gasto'],
               color=CIUDAD_COLORS[ciudad], s=100, zorder=3,
               label=ciudad, edgecolors='white', linewidths=0.5)

ax.set_xlabel('Días de estadía', color=WHITE, fontsize=13)
ax.set_ylabel('Gasto turístico (miles COP)', color=WHITE, fontsize=13)
ax.set_title('Gasto vs días de estadía por ciudad\n'
             '(n = 5 turistas por ciudad)', color=WHITE, fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 9)

# Anotación
ax.text(0.02, 0.97,
        'Cartagena gasta más.\nBogotá gasta menos.\n¿Cómo modelamos esto?',
        transform=ax.transAxes, color='lightgray', fontsize=9,
        va='top', style='italic')

plt.tight_layout()
plt.savefig('colombia_datos.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


### Las tres estrategias

| Estrategia | Descripción | Problema |
|---|---|---|
| **Complete Pooling** | Una sola recta para todos | Ignora diferencias entre ciudades |
| **No Pooling** | Recta separada por ciudad | Inestable con n=5, no predice Cali |
| **Jerárquico** | Rectas conectadas por distribución común | — |


In [ ]:
# ── Modelo 1: Complete Pooling ───────────────────────────────────────────────
with pm.Model() as modelo_pooling:
    mu    = pm.Normal('mu',   mu=600, sigma=300)
    beta  = pm.Normal('beta', mu=100, sigma=100)
    sigma = pm.Exponential('sigma', lam=0.005)
    gasto_mu = mu + beta * turismo['dias'].values
    y = pm.Normal('y', mu=gasto_mu, sigma=sigma, observed=turismo['gasto'].values)
    idata_pooling = pm.sample(1000, tune=1000, random_seed=SEED,
                              progressbar=False, chains=4)

b0_pool = float(idata_pooling.posterior['mu'].mean())
b1_pool = float(idata_pooling.posterior['beta'].mean())
print(f'Complete Pooling — μ={b0_pool:.0f}  β={b1_pool:.1f}')
print('Una sola recta: ignora que Cartagena y Bogotá son muy distintas.')


In [ ]:
# ── Modelo 2: No Pooling ─────────────────────────────────────────────────────
with pm.Model() as modelo_no_pool:
    mu_j   = pm.Normal('mu_j',   mu=600, sigma=300, shape=N_CIUDADES)
    beta_j = pm.Normal('beta_j', mu=100, sigma=100, shape=N_CIUDADES)
    sigma  = pm.Exponential('sigma', lam=0.005)
    gasto_mu = mu_j[ciudad_idx] + beta_j[ciudad_idx] * turismo['dias'].values
    y = pm.Normal('y', mu=gasto_mu, sigma=sigma, observed=turismo['gasto'].values)
    idata_no_pool = pm.sample(1000, tune=1000, random_seed=SEED,
                              progressbar=False, chains=4)

mu_j_np   = idata_no_pool.posterior['mu_j'].mean(dim=['chain','draw']).values
beta_j_np = idata_no_pool.posterior['beta_j'].mean(dim=['chain','draw']).values
print('No Pooling — estimaciones por ciudad:')
for i, c in enumerate(ciudades):
    print(f'  {c:12s}: μ={mu_j_np[i]:.0f}  β={beta_j_np[i]:.1f}')
print('\n⚠ Cali: NO tiene parámetros. El modelo no puede predecir.')


In [ ]:
# ── Modelo 3: Jerárquico (Partial Pooling) ───────────────────────────────────
with pm.Model() as modelo_hier:
    # Nivel 2 — hiperparámetros (distribución nacional)
    mu_0     = pm.Normal('mu_0',    mu=600, sigma=300)
    beta_0   = pm.Normal('beta_0',  mu=100, sigma=100)
    sigma_mu = pm.Exponential('sigma_mu',   lam=0.005)
    sigma_b  = pm.Exponential('sigma_beta', lam=0.02)

    # Nivel 1 — parámetros por ciudad, vienen de la distribución nacional
    mu_j   = pm.Normal('mu_j',   mu=mu_0,  sigma=sigma_mu, shape=N_CIUDADES)
    beta_j = pm.Normal('beta_j', mu=beta_0, sigma=sigma_b,  shape=N_CIUDADES)
    sigma  = pm.Exponential('sigma', lam=0.005)

    gasto_mu = mu_j[ciudad_idx] + beta_j[ciudad_idx] * turismo['dias'].values
    y = pm.Normal('y', mu=gasto_mu, sigma=sigma, observed=turismo['gasto'].values)
    idata_hier = pm.sample(2000, tune=1000, target_accept=0.9,
                           random_seed=SEED, progressbar=False, chains=4)

mu_j_h   = idata_hier.posterior['mu_j'].mean(dim=['chain','draw']).values
beta_j_h = idata_hier.posterior['beta_j'].mean(dim=['chain','draw']).values
mu_0_h   = float(idata_hier.posterior['mu_0'].mean())
print(f'Jerárquico — μ₀ global = {mu_0_h:.0f}')
print('Estimaciones por ciudad (con shrinkage):')
for i, c in enumerate(ciudades):
    print(f'  {c:12s}: μ={mu_j_h[i]:.0f}  β={beta_j_h[i]:.1f}')

# Diagnósticos rápidos
diag = az.summary(idata_hier, var_names=['mu_0','beta_0','sigma_mu','sigma_beta'], round_to=2)
print(f'\nR-hat máximo: {diag["r_hat"].max():.3f}  (debe ser < 1.01)')


In [ ]:
# ── Gráfico comparativo: las 3 estrategias ───────────────────────────────────
dias_range = np.linspace(1, 8, 80)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
titulos = ['Complete Pooling\nuna recta para todos',
           'No Pooling\nrecta separada por ciudad',
           'Jerárquico\nconectado por distribución común']

for ax, titulo in zip(axes, titulos):
    for ciudad in ciudades:
        subset = turismo[turismo['ciudad'] == ciudad]
        ax.scatter(subset['dias'], subset['gasto'],
                   color=CIUDAD_COLORS[ciudad], s=70, zorder=3,
                   edgecolors='white', linewidths=0.4)
    ax.set_title(titulo, color=WHITE, fontsize=11)
    ax.set_xlabel('Días de estadía', color=WHITE)
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 2400)

axes[0].set_ylabel('Gasto (miles COP)', color=WHITE)

# Panel 1: Complete Pooling — una sola recta
axes[0].plot(dias_range, b0_pool + b1_pool * dias_range,
             color='white', linewidth=2.5, label='Recta única')
axes[0].legend(fontsize=9)

# Panel 2: No Pooling — 3 rectas independientes
for i, ciudad in enumerate(ciudades):
    axes[1].plot(dias_range, mu_j_np[i] + beta_j_np[i] * dias_range,
                 color=CIUDAD_COLORS[ciudad], linewidth=2, label=ciudad)
axes[1].legend(fontsize=9)

# Panel 3: Jerárquico — 3 rectas moderadas
for i, ciudad in enumerate(ciudades):
    axes[2].plot(dias_range, mu_j_h[i] + beta_j_h[i] * dias_range,
                 color=CIUDAD_COLORS[ciudad], linewidth=2, label=ciudad)
# Media global
b1_h = float(idata_hier.posterior['beta_0'].mean())
axes[2].plot(dias_range, mu_0_h + b1_h * dias_range,
             color='white', linewidth=1.5, linestyle='--', alpha=0.5, label='Media global')
axes[2].legend(fontsize=9)

plt.suptitle('Tres estrategias para el mismo problema — Turismo en Colombia',
             color=WHITE, fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('colombia_tres_estrategias.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


In [ ]:
# ── Dumbbell plot: Shrinkage en acción ───────────────────────────────────────
# Medias brutas por ciudad
raw_means = turismo.groupby('ciudad_id')['gasto'].mean().values

fig, ax = plt.subplots(figsize=(9, 4))

y_pos = np.arange(N_CIUDADES)

# Líneas de conexión (shrinkage)
for i in range(N_CIUDADES):
    ax.plot([raw_means[i], mu_j_h[i]], [i, i],
            color='white', alpha=0.4, linewidth=2, zorder=1)

# No pooling (media bruta)
ax.scatter(raw_means, y_pos, color=ORANGE, s=120, zorder=3,
           label='No Pooling (media bruta)', edgecolors='white', linewidths=0.5)

# Jerárquico
ax.scatter(mu_j_h, y_pos, color=BLUE, s=120, zorder=4,
           label='Jerárquico (partial pooling)', edgecolors='white', linewidths=0.5)

# Media global
ax.axvline(mu_0_h, color='white', linewidth=1.5, linestyle='--',
           alpha=0.6, label=f'Media global (μ₀ = {mu_0_h:.0f})')

# Flechas de shrinkage
for i, ciudad in enumerate(ciudades):
    diff = mu_j_h[i] - raw_means[i]
    ax.annotate(f'Δ={diff:+.0f}',
                xy=(mu_j_h[i], i), xytext=(mu_j_h[i] + 30, i + 0.25),
                color='lightgray', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(ciudades, fontsize=12)
ax.set_xlabel('Gasto base estimado (miles COP)', color=WHITE, fontsize=12)
ax.set_title('Shrinkage: las estimaciones jerárquicas se acercan al promedio nacional\n'
             '(con n=5 por ciudad, el efecto es claro)', color=WHITE, fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('colombia_shrinkage_dumbbell.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


In [ ]:
# ── Predicción para Cali (ciudad nueva sin datos) ────────────────────────────
post = idata_hier.posterior
mu_0_s    = post['mu_0'].values.flatten()
sigma_mu_s = post['sigma_mu'].values.flatten()
beta_0_s  = post['beta_0'].values.flatten()
sigma_b_s = post['sigma_beta'].values.flatten()
sigma_s   = post['sigma'].values.flatten()

rng_local = np.random.default_rng(SEED)
DIAS_PRED = 4  # turista con 4 días de estadía

# Cali: samplear parámetros de la distribución nacional (sin datos propios)
mu_cali   = rng_local.normal(mu_0_s,   sigma_mu_s)
beta_cali = rng_local.normal(beta_0_s, sigma_b_s)
pred_cali = rng_local.normal(mu_cali + beta_cali * DIAS_PRED, sigma_s)

# Ciudades conocidas: usar posterior propio
preds_conocidas = {}
for i, ciudad in enumerate(ciudades):
    mu_j_s   = post['mu_j'].values[:, :, i].flatten()
    beta_j_s = post['beta_j'].values[:, :, i].flatten()
    preds_conocidas[ciudad] = rng_local.normal(
        mu_j_s + beta_j_s * DIAS_PRED, sigma_s
    )

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 2500, 60)

for ciudad in ciudades:
    ax.hist(preds_conocidas[ciudad], bins=bins,
            color=CIUDAD_COLORS[ciudad], alpha=0.5, label=ciudad, edgecolor='none')
    ax.axvline(preds_conocidas[ciudad].mean(),
               color=CIUDAD_COLORS[ciudad], linewidth=1.5, linestyle='--')

ax.hist(pred_cali, bins=bins, color=ORANGE, alpha=0.45,
        label='Cali (sin datos — más incertidumbre)', edgecolor='none')
ax.axvline(pred_cali.mean(), color=ORANGE, linewidth=2, linestyle='--')

ax.set_xlabel(f'Gasto predicho para {DIAS_PRED} días de estadía (miles COP)',
              color=WHITE, fontsize=12)
ax.set_ylabel('Frecuencia', color=WHITE)
ax.set_title('Predicción jerárquica: ciudades conocidas vs Cali (nueva)\n'
             'El modelo puede predecir Cali usando la distribución nacional',
             color=WHITE, fontsize=13)
ax.legend(fontsize=10)

print(f'Predicciones para {DIAS_PRED} días de estadía:')
for ciudad in ciudades:
    p = preds_conocidas[ciudad]
    hdi = az.hdi(p, hdi_prob=0.9)
    print(f'  {ciudad:12s}: media={p.mean():.0f}  HDI90%=[{hdi[0]:.0f}, {hdi[1]:.0f}]')
hdi_c = az.hdi(pred_cali, hdi_prob=0.9)
print(f'  {"Cali":12s}: media={pred_cali.mean():.0f}  HDI90%=[{hdi_c[0]:.0f}, {hdi_c[1]:.0f}]  ← más ancho')

plt.tight_layout()
plt.savefig('colombia_prediccion_cali.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


### Resumen del ejemplo Colombia

| Estrategia | ¿Puede predecir Cali? | Incertidumbre | Problema |
|---|---|---|---|
| Complete Pooling | Sí, pero mal | Subestimada | Ignora diferencias entre ciudades |
| No Pooling | **No** | Alta por grupo | Sin datos = sin parámetros |
| **Jerárquico** | **Sí, con honestidad** | **Apropiada** | **Ninguno** |

**El modelo jerárquico no es más complicado — es más honesto sobre la estructura real de los datos.**

---


---
## PARTE 1: Dataset Spotify — Popularidad por artista
### (Capítulo 16 — Modelos jerárquicos sin predictores)

**Estructura de datos:**
- 44 artistas, cada uno con un número variable de canciones
- Popularidad de cada canción: escala 0–100
- Total: ~350 canciones

In [ ]:
# ── Generar datos sintéticos Spotify ──────────────────────────────────────
np.random.seed(SEED)

N_ARTISTS   = 44
TRUE_MU     = 52.0   # popularidad media global
TRUE_SIGMA_MU = 12.0 # variabilidad entre artistas
TRUE_SIGMA_Y  = 14.0 # variabilidad dentro del artista

# Número de canciones por artista (heterogéneo — como en el libro)
songs_per_artist = np.random.choice([2, 3, 4, 5, 6, 8, 10, 15, 20, 25], 
                                     size=N_ARTISTS, 
                                     p=[0.12, 0.15, 0.15, 0.15, 0.12, 0.10, 0.08, 0.07, 0.04, 0.02])

# Parámetros verdaderos por artista (μ_j)
mu_j_true = np.random.normal(TRUE_MU, TRUE_SIGMA_MU, N_ARTISTS)
mu_j_true = np.clip(mu_j_true, 10, 95)  # mantener en rango realista

# Generar canciones
artist_ids   = []
popularities = []
for j, (n_songs, mu_j) in enumerate(zip(songs_per_artist, mu_j_true)):
    pops = np.random.normal(mu_j, TRUE_SIGMA_Y, n_songs)
    pops = np.clip(pops, 0, 100)
    popularities.extend(pops)
    artist_ids.extend([j] * n_songs)

spotify = pd.DataFrame({
    'artist_id': artist_ids,
    'popularity': popularities
})
spotify['artist'] = spotify['artist_id'].apply(lambda x: f'Artist_{x:02d}')

print(f'Total canciones: {len(spotify)}')
print(f'Canciones por artista — min: {songs_per_artist.min()}, max: {songs_per_artist.max()}, media: {songs_per_artist.mean():.1f}')
spotify.head(8)

In [ ]:
# ── Exploración visual ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución global de popularidad
ax = axes[0]
ax.hist(spotify['popularity'], bins=25, color=PURPLE, alpha=0.7, edgecolor='none')
ax.axvline(spotify['popularity'].mean(), color=BLUE, linewidth=2, linestyle='--', label=f'Media = {spotify["popularity"].mean():.1f}')
ax.set_title('Distribución de popularidad (todas las canciones)', color=WHITE)
ax.set_xlabel('Popularidad (0–100)', color=WHITE)
ax.set_ylabel('Frecuencia', color=WHITE)
ax.legend()

# Media por artista — solo primeros 20
ax = axes[1]
artist_means = spotify.groupby('artist_id')['popularity'].agg(['mean', 'count']).reset_index()
artist_means = artist_means.sort_values('mean').head(20)
colors = [BLUE if c >= 8 else ORANGE if c >= 4 else GREEN 
          for c in artist_means['count']]
bars = ax.barh(range(len(artist_means)), artist_means['mean'], color=colors, alpha=0.8)
ax.axvline(spotify['popularity'].mean(), color=WHITE, linewidth=1.5, linestyle='--', alpha=0.6, label='Media global')
ax.set_yticks(range(len(artist_means)))
ax.set_yticklabels([f"A{int(r['artist_id']):02d} (n={int(r['count'])})" for _, r in artist_means.iterrows()], fontsize=7)
ax.set_title('Popularidad media por artista (primeros 20)', color=WHITE)
ax.set_xlabel('Popularidad media', color=WHITE)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=BLUE, label='n≥8'), Patch(facecolor=ORANGE, label='n=4–7'), Patch(facecolor=GREEN, label='n≤3')]
ax.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plt.savefig('shrinkage_data.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

### Modelo 1: Complete Pooling
Un solo parámetro $\mu$ para todos los artistas. Ignora la estructura de grupos.

In [ ]:
with pm.Model() as model_pooled:
    # Un solo prior para la media global
    mu    = pm.Normal('mu', mu=50, sigma=25)
    sigma = pm.Exponential('sigma', lam=1)
    
    # Likelihood — todas las observaciones del mismo mu
    y = pm.Normal('y', mu=mu, sigma=sigma, observed=spotify['popularity'].values)
    
    idata_pooled = pm.sample(1000, tune=1000, random_seed=SEED, progressbar=True)

print('\n--- Complete Pooling ---')
print(az.summary(idata_pooled, var_names=['mu', 'sigma'], round_to=2))

### Modelo 2: No Pooling
Un $\mu_j$ independiente para cada artista. No comparte información entre grupos.

In [ ]:
artist_idx = spotify['artist_id'].values.astype(int)

with pm.Model() as model_no_pool:
    # Prior independiente para cada artista
    mu_j  = pm.Normal('mu_j', mu=50, sigma=25, shape=N_ARTISTS)
    sigma = pm.Exponential('sigma', lam=1)
    
    y = pm.Normal('y', mu=mu_j[artist_idx], sigma=sigma, observed=spotify['popularity'].values)
    
    idata_no_pool = pm.sample(1000, tune=1000, random_seed=SEED, progressbar=True)

print('\n--- No Pooling (primeros 5 artistas) ---')
summary_np = az.summary(idata_no_pool, var_names=['mu_j'], round_to=2)
print(summary_np.head())

### Modelo 3: Jerárquico (Partial Pooling)

$$Y_{ij} \mid \mu_j, \sigma_y \sim \mathcal{N}(\mu_j, \sigma_y^2)$$
$$\mu_j \mid \mu, \sigma_\mu \sim \mathcal{N}(\mu, \sigma_\mu^2)$$
$$\mu \sim \mathcal{N}(50, 25^2), \quad \sigma_y \sim \text{Exp}(1), \quad \sigma_\mu \sim \text{Exp}(1)$$

In [ ]:
with pm.Model() as model_hier:
    # Nivel 2: hiperparámetros (distribución de los artistas)
    mu_global = pm.Normal('mu_global', mu=50, sigma=25)
    sigma_mu  = pm.Exponential('sigma_mu', lam=1)   # variabilidad entre artistas
    
    # Nivel 1: media de cada artista — conectada al nivel 2
    mu_j  = pm.Normal('mu_j', mu=mu_global, sigma=sigma_mu, shape=N_ARTISTS)
    sigma_y = pm.Exponential('sigma_y', lam=1)       # variabilidad dentro del artista
    
    # Likelihood
    y = pm.Normal('y', mu=mu_j[artist_idx], sigma=sigma_y, observed=spotify['popularity'].values)
    
    idata_hier = pm.sample(2000, tune=1000, random_seed=SEED, 
                           target_accept=0.9, progressbar=True)

print('\n--- Jerárquico (hiperparámetros) ---')
print(az.summary(idata_hier, var_names=['mu_global', 'sigma_mu', 'sigma_y'], round_to=2))

### Diagnósticos MCMC

In [ ]:
# R-hat y ESS
summary_hier = az.summary(idata_hier, var_names=['mu_global', 'sigma_mu', 'sigma_y'])
print('Diagnósticos del modelo jerárquico:')
print(summary_hier[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']].round(3))
print(f'\n¿Todos los R-hat < 1.01? {(summary_hier["r_hat"] < 1.01).all()}')

In [ ]:
# Trace plots de los hiperparámetros
az.plot_trace(idata_hier, var_names=['mu_global', 'sigma_mu', 'sigma_y'],
              figsize=(12, 6))
plt.suptitle('Trace plots — Modelo Jerárquico (Spotify)', color=WHITE, y=1.01)
plt.tight_layout()
plt.savefig('trace_plots.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

### Shrinkage: la magia del modelo jerárquico

In [ ]:
# Extraer estimaciones
mu_j_hier_post   = idata_hier.posterior['mu_j'].values.reshape(-1, N_ARTISTS)
mu_j_hier_mean   = mu_j_hier_post.mean(axis=0)

mu_j_np_post     = idata_no_pool.posterior['mu_j'].values.reshape(-1, N_ARTISTS)
mu_j_np_mean     = mu_j_np_post.mean(axis=0)

mu_global_mean   = idata_hier.posterior['mu_global'].values.mean()

# Media de los datos brutos por artista
raw_means = spotify.groupby('artist_id')['popularity'].mean().values

# Ordenar por media bruta
order = np.argsort(raw_means)

fig, ax = plt.subplots(figsize=(13, 6))

y_pos = np.arange(N_ARTISTS)

# Líneas de shrinkage
for i, j in enumerate(order):
    ax.plot([raw_means[j], mu_j_hier_mean[j]], [i, i], 
            color='white', alpha=0.15, linewidth=1.2)

# Puntos no-pooling (medias brutas)
ax.scatter(raw_means[order], y_pos, color=ORANGE, s=40, zorder=3, 
           label='No Pooling (media bruta)', alpha=0.85)

# Puntos jerárquicos
ax.scatter(mu_j_hier_mean[order], y_pos, color=BLUE, s=40, zorder=4,
           label='Jerárquico (partial pooling)', alpha=0.9)

# Media global
ax.axvline(mu_global_mean, color=WHITE, linewidth=2, linestyle='--', 
           alpha=0.6, label=f'Media global (μ = {mu_global_mean:.1f})')

# Tamaño de muestra por artista
for i, j in enumerate(order):
    n = songs_per_artist[j]
    ax.annotate(f'n={n}', (mu_j_hier_mean[j] + 0.5, i), 
               fontsize=5.5, color='gray', va='center')

ax.set_xlabel('Popularidad estimada', color=WHITE, fontsize=12)
ax.set_ylabel('Artista (ordenado por media bruta)', color=WHITE, fontsize=12)
ax.set_title('Shrinkage: el modelo jerárquico jala hacia la media global\n'
             '(grupos pequeños se acercan más)', color=WHITE, fontsize=13)
ax.legend(fontsize=9)
ax.set_yticks([])

plt.tight_layout()
plt.savefig('shrinkage.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# Correlación intragrupo
sigma_mu_samples = idata_hier.posterior['sigma_mu'].values.flatten()
sigma_y_samples  = idata_hier.posterior['sigma_y'].values.flatten()
rho_samples      = sigma_mu_samples**2 / (sigma_mu_samples**2 + sigma_y_samples**2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rho_samples, bins=40, color=PURPLE, alpha=0.8, edgecolor='none')
ax.axvline(rho_samples.mean(), color=BLUE, linewidth=2, linestyle='--',
           label=f'Media = {rho_samples.mean():.3f}')
ax.axvline(np.percentile(rho_samples, 2.5),  color=WHITE, linewidth=1, alpha=0.5)
ax.axvline(np.percentile(rho_samples, 97.5), color=WHITE, linewidth=1, alpha=0.5)
ax.set_title('Distribución posterior de la correlación intragrupo (ρ)\n'
             r'$\rho = \sigma_\mu^2 / (\sigma_\mu^2 + \sigma_y^2)$', color=WHITE)
ax.set_xlabel('ρ', color=WHITE)
ax.set_ylabel('Frecuencia', color=WHITE)
ax.legend()
print(f'ρ medio: {rho_samples.mean():.3f}  [HDI 95%: {np.percentile(rho_samples, 2.5):.3f}, {np.percentile(rho_samples, 97.5):.3f}]')
print('Interpretación: los artistas explican ~{:.0f}% de la variabilidad total'.format(rho_samples.mean()*100))
plt.tight_layout()
plt.show()

---
## PARTE 2: Dataset Running — Tiempos de carrera vs. edad
### (Capítulo 17 — Modelos jerárquicos con predictores)

**Estructura de datos:**
- 36 corredores, edades 50–60 años
- 252 observaciones totales (~7 por corredor)
- Variable respuesta: tiempo neto de carrera (minutos)
- Predictor: edad al momento de correr

In [ ]:
# ── Generar datos sintéticos Running ─────────────────────────────────────
np.random.seed(SEED + 1)

N_RUNNERS = 36

# Parámetros verdaderos del modelo generador
BETA_0_TRUE = 85.0   # tiempo base global (minutos)
BETA_1_TRUE = 1.2    # minutos adicionales por año de edad
SIGMA_0_TRUE = 12.0  # variabilidad en velocidad base entre corredores
SIGMA_1_TRUE = 0.4   # variabilidad en tasa de envejecimiento
SIGMA_Y_TRUE = 5.0   # ruido por carrera
RHO_TRUE = 0.3       # correlación entre intercept y slope

# Interceptos y slopes por corredor (bivariado)
cov_matrix = np.array([
    [SIGMA_0_TRUE**2, RHO_TRUE * SIGMA_0_TRUE * SIGMA_1_TRUE],
    [RHO_TRUE * SIGMA_0_TRUE * SIGMA_1_TRUE, SIGMA_1_TRUE**2]
])
runner_params = np.random.multivariate_normal(
    [BETA_0_TRUE, BETA_1_TRUE], cov_matrix, N_RUNNERS
)
beta_0j = runner_params[:, 0]
beta_1j = runner_params[:, 1]

# Generar observaciones
runner_ids = []
ages       = []
net_times  = []

for j in range(N_RUNNERS):
    n_obs  = np.random.choice([5, 6, 7, 8, 9, 10], p=[0.1, 0.2, 0.3, 0.2, 0.15, 0.05])
    age_j  = np.random.uniform(50, 61, n_obs)  # edad varía entre carreras
    net_j  = beta_0j[j] + beta_1j[j] * (age_j - 54) + np.random.normal(0, SIGMA_Y_TRUE, n_obs)
    runner_ids.extend([j] * n_obs)
    ages.extend(age_j)
    net_times.extend(net_j)

running = pd.DataFrame({
    'runner_id': runner_ids,
    'age':       ages,
    'net':       net_times
})

# Centrar edad (importante para interpretación del intercepto)
AGE_CENTER = running['age'].mean()
running['age_c'] = running['age'] - AGE_CENTER

runner_idx = running['runner_id'].values.astype(int)

print(f'Total observaciones: {len(running)}')
print(f'Corredores: {N_RUNNERS}')
print(f'Edad media (centro): {AGE_CENTER:.1f} años')
running.head(8)

In [ ]:
# ── Exploración visual ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter: edad vs. tiempo (coloreado por corredor)
ax = axes[0]
colors_runners = plt.cm.plasma(np.linspace(0.1, 0.9, N_RUNNERS))
for j in range(N_RUNNERS):
    mask = running['runner_id'] == j
    ax.scatter(running.loc[mask, 'age'], running.loc[mask, 'net'],
               color=colors_runners[j], alpha=0.5, s=20)
ax.set_xlabel('Edad (años)', color=WHITE)
ax.set_ylabel('Tiempo neto (min)', color=WHITE)
ax.set_title('Edad vs. Tiempo por corredor', color=WHITE)

# Distribución de tiempos
ax = axes[1]
ax.hist(running['net'], bins=30, color=BLUE, alpha=0.7, edgecolor='none')
ax.set_xlabel('Tiempo neto (min)', color=WHITE)
ax.set_ylabel('Frecuencia', color=WHITE)
ax.set_title('Distribución de tiempos de carrera', color=WHITE)

plt.tight_layout()
plt.show()

### Modelo A: Regresión Pooled
Una sola recta de regresión para todos los corredores.

In [ ]:
with pm.Model() as model_pooled_reg:
    beta_0 = pm.Normal('beta_0', mu=85, sigma=20)
    beta_1 = pm.Normal('beta_1', mu=0,  sigma=5)
    sigma  = pm.Exponential('sigma', lam=0.2)
    
    mu_net = beta_0 + beta_1 * running['age_c'].values
    y = pm.Normal('y', mu=mu_net, sigma=sigma, observed=running['net'].values)
    
    idata_pooled_reg = pm.sample(1000, tune=1000, random_seed=SEED, progressbar=True)

print('\n--- Pooled Regression ---')
print(az.summary(idata_pooled_reg, var_names=['beta_0', 'beta_1', 'sigma'], round_to=2))

### Modelo B: Random Intercepts
Cada corredor tiene su propio tiempo base, pero todos envejecen igual.

$$Y_{ij} \sim \mathcal{N}(\beta_{0j} + \beta_1 \cdot \text{age}_c, \sigma^2)$$
$$\beta_{0j} \sim \mathcal{N}(\beta_0, \sigma_0^2)$$

In [ ]:
with pm.Model() as model_ri:
    # Hiperparámetros del intercepto
    beta_0   = pm.Normal('beta_0', mu=85, sigma=20)
    sigma_0  = pm.Exponential('sigma_0', lam=0.1)  # variabilidad entre corredores
    
    # Intercepto por corredor
    beta_0j  = pm.Normal('beta_0j', mu=beta_0, sigma=sigma_0, shape=N_RUNNERS)
    
    # Pendiente global de edad
    beta_1   = pm.Normal('beta_1', mu=0, sigma=5)
    sigma_y  = pm.Exponential('sigma_y', lam=0.2)
    
    mu_net = beta_0j[runner_idx] + beta_1 * running['age_c'].values
    y = pm.Normal('y', mu=mu_net, sigma=sigma_y, observed=running['net'].values)
    
    idata_ri = pm.sample(2000, tune=1000, random_seed=SEED, 
                         target_accept=0.9, progressbar=True)

print('\n--- Random Intercepts ---')
print(az.summary(idata_ri, var_names=['beta_0', 'beta_1', 'sigma_0', 'sigma_y'], round_to=2))

### Modelo C: Random Intercepts + Slopes
Cada corredor tiene su propio tiempo base **y** su propia tasa de envejecimiento.

$$\begin{pmatrix}\beta_{0j}\\ \beta_{1j}\end{pmatrix} \sim \mathcal{N}_2\left(\begin{pmatrix}\beta_0\\ \beta_1\end{pmatrix}, \Sigma\right)$$

In [ ]:
with pm.Model() as model_rs:
    # Medias globales
    beta_0 = pm.Normal('beta_0', mu=85, sigma=20)
    beta_1 = pm.Normal('beta_1', mu=0,  sigma=5)
    
    # Desviaciones estándar
    sigma_0 = pm.Exponential('sigma_0', lam=0.1)
    sigma_1 = pm.Exponential('sigma_1', lam=1.0)
    
    # Correlación entre intercepto y slope
    rho_raw = pm.Beta('rho_raw', alpha=2, beta=2)  # Prior en [0,1]
    rho = pm.Deterministic('rho', 2 * rho_raw - 1)  # Transformar a [-1,1]
    
    # Matriz de covarianza
    cov = pm.math.stack([
        pm.math.stack([sigma_0**2, rho * sigma_0 * sigma_1]),
        pm.math.stack([rho * sigma_0 * sigma_1, sigma_1**2])
    ])
    
    # Efectos aleatorios bivariados
    runner_effects = pm.MvNormal('runner_effects', mu=[0, 0], cov=cov, shape=(N_RUNNERS, 2))
    beta_0j = pm.Deterministic('beta_0j', beta_0 + runner_effects[:, 0])
    beta_1j = pm.Deterministic('beta_1j', beta_1 + runner_effects[:, 1])
    
    sigma_y = pm.Exponential('sigma_y', lam=0.2)
    
    mu_net = beta_0j[runner_idx] + beta_1j[runner_idx] * running['age_c'].values
    y = pm.Normal('y', mu=mu_net, sigma=sigma_y, observed=running['net'].values)
    
    idata_rs = pm.sample(2000, tune=1500, random_seed=SEED, 
                         target_accept=0.92, progressbar=True)

print('\n--- Random Intercepts + Slopes ---')
print(az.summary(idata_rs, var_names=['beta_0', 'beta_1', 'sigma_0', 'sigma_1', 'rho', 'sigma_y'], round_to=3))

### Visualización: las tres rectas de regresión

In [ ]:
age_range = np.linspace(running['age'].min(), running['age'].max(), 100)
age_c_range = age_range - AGE_CENTER

# Posterior means
b0_pooled = idata_pooled_reg.posterior['beta_0'].values.mean()
b1_pooled = idata_pooled_reg.posterior['beta_1'].values.mean()

b0j_ri    = idata_ri.posterior['beta_0j'].values.mean(axis=(0,1))
b1_ri     = idata_ri.posterior['beta_1'].values.mean()

b0j_rs    = idata_rs.posterior['beta_0j'].values.mean(axis=(0,1))
b1j_rs    = idata_rs.posterior['beta_1j'].values.mean(axis=(0,1))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, title, color in zip(axes, 
    ['Pooled', 'Random Intercepts', 'Random Intercepts + Slopes'],
    [PURPLE, BLUE, GREEN]):
    # Datos
    ax.scatter(running['age'], running['net'], alpha=0.25, s=12, color='white')
    ax.set_title(title, color=WHITE, fontsize=11)
    ax.set_xlabel('Edad (años)', color=WHITE)

axes[0].set_ylabel('Tiempo neto (min)', color=WHITE)

# Pooled: una sola recta
axes[0].plot(age_range, b0_pooled + b1_pooled * age_c_range, color=PURPLE, linewidth=2.5)

# Random Intercepts: misma pendiente, intercepts distintos
for j in range(N_RUNNERS):
    axes[1].plot(age_range, b0j_ri[j] + b1_ri * age_c_range, 
                 color=BLUE, alpha=0.25, linewidth=1)
axes[1].plot(age_range, b0j_ri.mean() + b1_ri * age_c_range, color=WHITE, linewidth=2.5, linestyle='--')

# Random Slopes: pendiente e intercept distintos
for j in range(N_RUNNERS):
    axes[2].plot(age_range, b0j_rs[j] + b1j_rs[j] * age_c_range, 
                 color=GREEN, alpha=0.25, linewidth=1)
axes[2].plot(age_range, b0j_rs.mean() + b1j_rs.mean() * age_c_range, color=WHITE, linewidth=2.5, linestyle='--')

plt.suptitle('Comparación de modelos — Datos de Running', color=WHITE, fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('regression_compare.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

### Comparación de modelos: LOO-CV

In [ ]:
# Calcular LOO para los tres modelos
loo_pooled = az.loo(idata_pooled_reg, pointwise=True)
loo_ri     = az.loo(idata_ri,         pointwise=True)
loo_rs     = az.loo(idata_rs,         pointwise=True)

print('=== Leave-One-Out Cross-Validation ===')
print(f'Pooled:          ELPD_LOO = {loo_pooled.elpd_loo:.1f}  (SE = {loo_pooled.se:.1f})')
print(f'Random Intercepts: ELPD_LOO = {loo_ri.elpd_loo:.1f}  (SE = {loo_ri.se:.1f})')
print(f'Random Slopes:   ELPD_LOO = {loo_rs.elpd_loo:.1f}  (SE = {loo_rs.se:.1f})')

# Tabla de comparación
comparison = az.compare({
    'Pooled':            idata_pooled_reg,
    'Random Intercepts': idata_ri,
    'Random Slopes':     idata_rs
})
print('\n=== Tabla comparativa (ordenada por ELPD) ===')
print(comparison[['elpd_loo', 'p_loo', 'elpd_diff', 'weight']].round(2))

### Predicción: corredor conocido vs. corredor nuevo

In [ ]:
# Corredor conocido: Runner #5, a los 57 años
KNOWN_RUNNER  = 5
AGE_PREDICT   = 57.0
age_c_predict = AGE_PREDICT - AGE_CENTER

# Posterior samples del modelo random intercepts
post = idata_ri.posterior
b0j_samples   = post['beta_0j'].values.reshape(-1, N_RUNNERS)
b1_samples    = post['beta_1'].values.flatten()
sigma_samples = post['sigma_y'].values.flatten()

# Predicción corredor conocido
mu_known    = b0j_samples[:, KNOWN_RUNNER] + b1_samples * age_c_predict
pred_known  = np.random.normal(mu_known, sigma_samples)

# Predicción corredor nuevo (usamos la distribución global)
b0_samples  = post['beta_0'].values.flatten()
s0_samples  = post['sigma_0'].values.flatten()
b0j_new     = np.random.normal(b0_samples, s0_samples)  # nuevo corredor
mu_new      = b0j_new + b1_samples * age_c_predict
pred_new    = np.random.normal(mu_new, sigma_samples)

fig, ax = plt.subplots(figsize=(9, 5))

# Distribuciones de predicción
bins = np.linspace(50, 130, 60)
ax.hist(pred_known, bins=bins, color=BLUE,   alpha=0.65, label=f'Corredor conocido (#{KNOWN_RUNNER})', edgecolor='none')
ax.hist(pred_new,   bins=bins, color=ORANGE, alpha=0.55, label='Corredor nuevo (desconocido)',         edgecolor='none')

# HDI 95%
hdi_known = az.hdi(pred_known, hdi_prob=0.95)
hdi_new   = az.hdi(pred_new,   hdi_prob=0.95)
ax.axvline(pred_known.mean(), color=BLUE,   linewidth=2, linestyle='--')
ax.axvline(pred_new.mean(),   color=ORANGE, linewidth=2, linestyle='--')

ax.set_xlabel('Tiempo neto predicho (min) a los 57 años', color=WHITE, fontsize=12)
ax.set_ylabel('Frecuencia', color=WHITE)
ax.set_title('Predicción: corredor conocido vs. nuevo\nEl corredor nuevo tiene más incertidumbre', color=WHITE, fontsize=12)
ax.legend(fontsize=10)

print(f'Corredor conocido: media = {pred_known.mean():.1f}  HDI95% = [{hdi_known[0]:.1f}, {hdi_known[1]:.1f}]  ancho = {hdi_known[1]-hdi_known[0]:.1f}')
print(f'Corredor nuevo:    media = {pred_new.mean():.1f}    HDI95% = [{hdi_new[0]:.1f}, {hdi_new[1]:.1f}]  ancho = {hdi_new[1]-hdi_new[0]:.1f}')

plt.tight_layout()
plt.savefig('posterior_prediction.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

---
## Resumen final

| Modelo | Datos | Ventaja principal |
|--------|-------|-------------------|
| Complete Pooling | Spotify | Simple, pero ignora diferencias entre artistas |
| No Pooling | Spotify | Flexible, pero sobreajusta grupos pequeños |
| **Jerárquico** | **Spotify** | **Balance: shrinkage hacia la media global** |
| Pooled Regression | Running | Una sola recta para todos |
| Random Intercepts | Running | Velocidad base distinta por corredor |
| **Random Slopes** | **Running** | **Cada corredor envejece a su propio ritmo** |

**Conclusión clave:** Los modelos jerárquicos no son más complicados — son más honestos sobre la estructura real de los datos.